# Monthly proxy pipeline

This notebook builds a longer-history monthly panel with simpler proxies and simpler code.

- Price response: USDA NASS monthly Prices Received
- USD index: FRED broad dollar series joined old-to-new
- Volatility proxy: monthly standard deviation of daily S&P 500 returns
- Climate proxies: NOAA Climate at a Glance regional monthly values
- Disaster counts: FEMA monthly declarations


In [5]:
import io
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Resolve the repo root whether the notebook is opened from the repo root or from scripts/.
cwd = Path.cwd().resolve()
if cwd.name == "scripts":
    ROOT = cwd.parent
elif (cwd / "data").exists() and (cwd / "scripts").exists():
    ROOT = cwd
else:
    ROOT = cwd.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

START_DATE = "1980-01-01"
START_YEAR = 1980

FRED_URL = "https://api.stlouisfed.org/fred/series/observations"
NASS_URL = "https://quickstats.nass.usda.gov/api/api_GET/"
NOAA_CAG_URL = "https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance"
FEMA_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
ONI_URL = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

# One climate region proxy per crop to keep the panel simple and reproducible.
COMMODITIES = [
    {"commodity": "corn", "nass_candidates": ["CORN"], "climate_location": 261},
    {"commodity": "soybean", "nass_candidates": ["SOYBEANS", "SOYBEAN"], "climate_location": 262},
    {"commodity": "wheat", "nass_candidates": ["WHEAT"], "climate_location": 256},
]

# Load API keys from the nearest .env file.
def load_dotenv():
    env_candidates = [ROOT / ".env", Path.cwd().resolve() / ".env", ROOT.parent / ".env"]
    for env_path in env_candidates:
        if not env_path.exists():
            continue
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
        return

# Normalize dates to month end so every source uses the same calendar.
def month_end(x):
    return pd.Timestamp(x).to_period("M").to_timestamp("M")

def last_complete_month_end():
    today = pd.Timestamp.today().normalize()
    return today.replace(day=1) - pd.offsets.Day(1)

def month_index(start_date, end_date):
    return pd.date_range(month_end(start_date), end_date, freq="ME")

def get_json(url, params, timeout=60):
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()

def get_text(url, timeout=60):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.text

def to_number(series):
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

load_dotenv()

# Avoid partial current-month data by stopping at the last completed month.
END_DATE = last_complete_month_end()
print("Building through", END_DATE.date())


Building through 2026-03-31


In [6]:
# FRED helper for monthly macro series.
def fetch_fred_series(series_id):
    api_key = os.getenv("FRED_API_KEY")
    if not api_key:
        raise RuntimeError("Missing FRED_API_KEY")
    payload = get_json(
        FRED_URL,
        {
            "series_id": series_id,
            "api_key": api_key,
            "file_type": "json",
            "observation_start": START_DATE,
            "observation_end": END_DATE.strftime("%Y-%m-%d"),
        },
    )
    df = pd.DataFrame(payload.get("observations", []))
    if df.empty:
        return pd.DataFrame(columns=["date", series_id])
    df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp("M")
    df[series_id] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", series_id]]

# Use the older broad dollar index first, then fall forward into the newer broad index.
usd_old = fetch_fred_series("TWEXBMTH").rename(columns={"TWEXBMTH": "usd_old"})
usd_new = fetch_fred_series("TWEXBGSMTH").rename(columns={"TWEXBGSMTH": "usd_new"})
interest_df = fetch_fred_series("TB3MS").rename(columns={"TB3MS": "interest_rate"})

usd_df = usd_old.merge(usd_new, on="date", how="outer").sort_values("date")
usd_df["usd_index"] = usd_df["usd_new"].combine_first(usd_df["usd_old"])
usd_df["usd_index_source"] = "FRED_TWEXBMTH_then_TWEXBGSMTH"

# Use FRED volatility series only, so the notebook does not depend on Yahoo rate limits.
vxo_df = fetch_fred_series("VXOCLS").rename(columns={"VXOCLS": "vxo"})
vix_df = fetch_fred_series("VIXCLS").rename(columns={"VIXCLS": "vix_new"})

vxo_df["date"] = pd.to_datetime(vxo_df["date"]).dt.to_period("M").dt.to_timestamp("M")
vix_df["date"] = pd.to_datetime(vix_df["date"]).dt.to_period("M").dt.to_timestamp("M")

vxo_df = vxo_df.groupby("date", as_index=False)["vxo"].mean()
vix_df = vix_df.groupby("date", as_index=False)["vix_new"].mean()

vol_df = vxo_df.merge(vix_df, on="date", how="outer").sort_values("date")
vol_df["vix"] = vol_df["vix_new"].combine_first(vol_df["vxo"])
vol_df["vix_source"] = "FRED_VXOCLS_then_VIXCLS"
vol_df = vol_df[["date", "vix", "vix_source"]]

macro_df = usd_df[["date", "usd_index", "usd_index_source"]].merge(interest_df, on="date", how="outer")
macro_df = macro_df.merge(vol_df, on="date", how="outer").sort_values("date").reset_index(drop=True)
macro_df.head()


,date,usd_index,usd_index_source,interest_rate,vix,vix_source
0,1980-01-31,35.8600,FRED_TWEXBMTH_then_TWEXBGSMTH,12.00,NaN,NaN
1,1980-02-29,36.2089,FRED_TWEXBMTH_then_TWEXBGSMTH,12.86,NaN,NaN
2,1980-03-31,37.1465,FRED_TWEXBMTH_then_TWEXBGSMTH,15.20,NaN,NaN
3,1980-04-30,37.4326,FRED_TWEXBMTH_then_TWEXBGSMTH,13.20,NaN,NaN
4,1980-05-31,36.2590,FRED_TWEXBMTH_then_TWEXBGSMTH,8.58,NaN,NaN


In [7]:
# USDA Quick Stats helper for monthly national prices received.
def fetch_nass_rows(candidate):
    api_key = os.getenv("USDA_NASS_API_KEY")
    if not api_key:
        raise RuntimeError("Missing USDA_NASS_API_KEY")
    payload = get_json(
        NASS_URL,
        {
            "key": api_key,
            "format": "JSON",
            "commodity_desc": candidate,
            "statisticcat_desc": "PRICE RECEIVED",
            "agg_level_desc": "NATIONAL",
            "freq_desc": "MONTHLY",
            "source_desc": "SURVEY",
            "year__GE": START_YEAR,
        },
    )
    df = pd.DataFrame(payload.get("data", []))
    if not df.empty:
        df.columns = [str(c).lower() for c in df.columns]
    return df

# Keep the cleanest monthly national series and convert it into a month-end time series.
def pick_nass_series(rows):
    if rows.empty:
        return rows
    rows = rows.copy()
    rows["price_proxy"] = to_number(rows["value"])
    rows = rows[rows["price_proxy"].notna()].copy()
    if rows.empty:
        return rows
    if "domain_desc" in rows.columns:
        total_rows = rows[rows["domain_desc"].fillna("").str.upper() == "TOTAL"].copy()
        if not total_rows.empty:
            rows = total_rows
    rows["month"] = pd.to_numeric(rows["begin_code"], errors="coerce")
    rows = rows[rows["month"].between(1, 12, inclusive="both")].copy()
    rows["year"] = pd.to_numeric(rows["year"], errors="coerce")
    rows["date"] = pd.to_datetime(dict(year=rows["year"].astype(int), month=rows["month"].astype(int), day=1)).dt.to_period("M").dt.to_timestamp("M")
    rows = rows.sort_values("date")
    rows = rows.drop_duplicates("date", keep="last")
    return rows

# Build one monthly price proxy series for each commodity.
price_frames = []
for spec in COMMODITIES:
    chosen = pd.DataFrame()
    chosen_name = None
    for candidate in spec["nass_candidates"]:
        temp = pick_nass_series(fetch_nass_rows(candidate))
        if not temp.empty:
            chosen = temp
            chosen_name = candidate
            break
    if chosen.empty:
        raise RuntimeError(f"No USDA monthly price series found for {spec['commodity']}")
    df = chosen[["date", "price_proxy"]].copy()
    df = df[df["date"] <= END_DATE].sort_values("date")
    df["commodity"] = spec["commodity"]
    df["price_source"] = "USDA_NASS_PRICE_RECEIVED"
    df["price_source_detail"] = chosen_name
    df["futures_price"] = df["price_proxy"]
    df["futures_return"] = np.log(df["futures_price"]).diff()
    price_frames.append(df)

price_df = pd.concat(price_frames, ignore_index=True).sort_values(["commodity", "date"]).reset_index(drop=True)
price_df.head()


,date,price_proxy,commodity,price_source,price_source_detail,futures_price,futures_return
0,1980-01-31,2.45,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.45,NaN
1,1980-02-29,2.39,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.39,-0.024795
2,1980-03-31,2.40,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.40,0.004175
3,1980-04-30,2.36,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.36,-0.016807
4,1980-05-31,2.42,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.42,0.025106


In [8]:
# ENSO ONI: reuse the same ASCII source and parsing approach as monthly_pipeline.ipynb.
def fetch_oni():
    oni = pd.read_csv(ONI_URL, sep=r"\s+")
    oni.columns = [str(c).strip() for c in oni.columns]
    required_cols = {"SEAS", "YR", "ANOM"}
    if not required_cols.issubset(set(oni.columns)):
        raise RuntimeError("Unexpected ONI columns: " + str(oni.columns.tolist()))

    month_map = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6, "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}
    oni = oni.rename(columns={"SEAS": "season", "YR": "year", "ANOM": "enso_oni"})
    oni["month"] = oni["season"].map(month_map)
    oni = oni.dropna(subset=["month"])
    oni["date"] = pd.to_datetime(dict(year=oni["year"].astype(int), month=oni["month"].astype(int), day=1)) + pd.offsets.MonthEnd(0)
    oni["enso_oni"] = pd.to_numeric(oni["enso_oni"], errors="coerce")
    oni = oni[(oni["date"] >= month_end(START_DATE)) & (oni["date"] <= END_DATE)]
    return oni[["date", "enso_oni"]].drop_duplicates("date").sort_values("date").reset_index(drop=True)

# Aggregate FEMA disaster declarations into monthly counts.
def fetch_fema():
    rows = []
    skip = 0
    end_ts = END_DATE.strftime("%Y-%m-%dT23:59:59.999Z")
    filters = f"declarationDate ge '{START_YEAR}-01-01T00:00:00.000Z' and declarationDate le '{end_ts}'"
    while True:
        payload = get_json(FEMA_URL, {"$select": "declarationDate,incidentType", "$filter": filters, "$top": 5000, "$skip": skip, "$orderby": "declarationDate asc", "$format": "json"})
        batch = payload.get("DisasterDeclarationsSummaries", [])
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < 5000:
            break
        skip += 5000
    fema = pd.DataFrame(rows)
    if fema.empty:
        calendar = pd.DataFrame({"date": month_index(START_DATE, END_DATE)})
        for col in ["disaster_event_count", "disaster_storm_count", "disaster_fire_count", "disaster_flood_count"]:
            calendar[col] = 0.0
        return calendar
    fema["date"] = pd.to_datetime(fema["declarationDate"], errors="coerce", utc=True).dt.tz_localize(None).dt.to_period("M").dt.to_timestamp("M")
    fema = fema.dropna(subset=["date"])
    fema["incidentType"] = fema["incidentType"].fillna("Unknown")
    monthly = fema.groupby("date").size().rename("disaster_event_count").reset_index()
    for label, out_col in [("Severe Storm(s)", "disaster_storm_count"), ("Fire", "disaster_fire_count"), ("Flood", "disaster_flood_count")]:
        temp = fema.assign(flag=fema["incidentType"].str.lower().eq(label.lower())).groupby("date")["flag"].sum().rename(out_col).reset_index()
        monthly = monthly.merge(temp, on="date", how="left")
    calendar = pd.DataFrame({"date": month_index(START_DATE, END_DATE)})
    monthly = calendar.merge(monthly, on="date", how="left")
    for col in ["disaster_event_count", "disaster_storm_count", "disaster_fire_count", "disaster_flood_count"]:
        monthly[col] = pd.to_numeric(monthly[col], errors="coerce").fillna(0.0)
    return monthly

# Parse NOAA Climate at a Glance responses from either CSV or JSON.
def parse_cag(response):
    text = response.text.strip()
    ctype = response.headers.get("Content-Type", "").lower()

    if "json" in ctype or text.startswith("{"):
        payload = response.json()
        data = payload.get("data", {}) if isinstance(payload, dict) else {}
        rows = []
        for date_key, value_obj in data.items():
            if isinstance(value_obj, dict):
                value = value_obj.get("value")
            else:
                value = value_obj
            rows.append({"date_raw": str(date_key), "value": value})
        df = pd.DataFrame(rows)
    else:
        df = pd.read_csv(io.StringIO(text), comment="#")
        df.columns = [str(c).strip().lower() for c in df.columns]
        if not {"date", "value"}.issubset(df.columns):
            raise ValueError("Unexpected NOAA CSV columns: " + str(df.columns.tolist()))
        df = df.rename(columns={"date": "date_raw", "value": "value"})

    if df.empty:
        return df

    df["date_raw"] = df["date_raw"].astype(str).str.strip()
    df["date"] = pd.to_datetime(df["date_raw"], format="%Y%m", errors="coerce").dt.to_period("M").dt.to_timestamp("M")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna(subset=["date", "value"])[["date", "value"]].drop_duplicates("date").sort_values("date").reset_index(drop=True)

def fetch_cag_series(location, parameter):
    month_frames = []
    errors = []

    for month in range(1, 13):
        url = f"{NOAA_CAG_URL}/regional/time-series/{location}/{parameter}/1/{month}/{START_YEAR}-{END_DATE.year}.csv"
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            df = parse_cag(response)
            if df.empty:
                errors.append(f"{url} -> empty response")
                continue
            df = df[df["date"].dt.month == month].copy()
            month_frames.append(df)
        except Exception as exc:
            errors.append(f"{url} -> {exc}")

    if not month_frames:
        raise RuntimeError("Unable to fetch NOAA Climate at a Glance series for " + str(location) + " " + str(parameter) + "\n" + "\n".join(errors))

    out = pd.concat(month_frames, ignore_index=True)
    out = out[(out["date"] >= month_end(START_DATE)) & (out["date"] <= END_DATE)]
    out = out.drop_duplicates("date").sort_values("date").reset_index(drop=True)
    return out

oni_df = fetch_oni()
fema_df = fetch_fema()

# Pull four simple monthly climate proxies for each crop region.
climate_frames = []
for spec in COMMODITIES:
    temperature = fetch_cag_series(spec["climate_location"], "tavg").rename(columns={"value": "temperature_anomaly"})
    precipitation = fetch_cag_series(spec["climate_location"], "pcp").rename(columns={"value": "precipitation_anomaly"})
    drought = fetch_cag_series(spec["climate_location"], "pdsi").rename(columns={"value": "drought_index"})
    heat = fetch_cag_series(spec["climate_location"], "tmax").rename(columns={"value": "extreme_heat_events"})
    df = temperature.merge(precipitation, on="date", how="outer")
    df = df.merge(drought, on="date", how="outer")
    df = df.merge(heat, on="date", how="outer")
    df = df.merge(oni_df, on="date", how="left")
    df = df.merge(fema_df, on="date", how="left")
    df["commodity"] = spec["commodity"]
    df["climate_location_code"] = spec["climate_location"]
    df = df[(df["date"] >= month_end(START_DATE)) & (df["date"] <= END_DATE)].sort_values("date").reset_index(drop=True)
    climate_frames.append(df)

climate_df = pd.concat(climate_frames, ignore_index=True).sort_values(["commodity", "date"]).reset_index(drop=True)
climate_df.head()



,date,temperature_anomaly,precipitation_anomaly,drought_index,extreme_heat_events,enso_oni,disaster_event_count,disaster_storm_count,disaster_fire_count,disaster_flood_count,commodity,climate_location_code
0,1980-01-31,21.6,1.01,2.99,29.8,0.59,0.0,0.0,0.0,0.0,corn,261
1,1980-02-29,20.3,0.91,2.65,29.2,0.46,17.0,0.0,0.0,14.0,corn,261
2,1980-03-31,31.8,2.21,2.65,41.3,0.34,0.0,0.0,0.0,0.0,corn,261
3,1980-04-30,48.6,1.88,-0.63,60.6,0.38,28.0,0.0,1.0,12.0,corn,261
4,1980-05-31,60.9,2.72,-1.18,73.8,0.48,61.0,0.0,0.0,6.0,corn,261


In [9]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Final monthly panel: price proxy by commodity, shared macro data, and commodity-specific climate proxies.
panel = price_df.merge(macro_df, on="date", how="left")
panel = panel.merge(climate_df, on=["date", "commodity"], how="left")
panel = panel.sort_values(["commodity", "date"]).reset_index(drop=True)

required_cols = [
    "futures_return",
    "usd_index",
    "interest_rate",
    "vix",
    "enso_oni",
    "temperature_anomaly",
    "precipitation_anomaly",
    "drought_index",
    "extreme_heat_events",
    "disaster_event_count",
    "disaster_storm_count",
    "disaster_fire_count",
    "disaster_flood_count",
]

first_valid = {}
for col in required_cols:
    dates = panel.loc[panel[col].notna(), "date"]
    if dates.empty:
        raise RuntimeError(f"Required column {col} has no non-null values")
    first_valid[col] = dates.min()

chosen_start = max(first_valid.values())
print("First available date by required column:")
for col, dt in first_valid.items():
    print(" ", col, dt)
print("Chosen panel start date:", chosen_start)

panel = panel[panel["date"] >= chosen_start].copy()

price_df.to_csv(RAW_DIR / "price_proxy_monthly.csv", index=False)
macro_df.to_csv(RAW_DIR / "macro_monthly.csv", index=False)
climate_df.to_csv(RAW_DIR / "climate_disaster_monthly.csv", index=False)
panel.to_csv(PROCESSED_DIR / "monthly_panel.csv", index=False)

print("Saved", RAW_DIR / "price_proxy_monthly.csv", price_df.shape)
print("Saved", RAW_DIR / "macro_monthly.csv", macro_df.shape)
print("Saved", RAW_DIR / "climate_disaster_monthly.csv", climate_df.shape)
print("Saved", PROCESSED_DIR / "monthly_panel.csv", panel.shape)
panel.head()


First available date by required column:
  futures_return 1980-02-29 00:00:00
  usd_index 1980-01-31 00:00:00
  interest_rate 1980-01-31 00:00:00
  vix 1986-01-31 00:00:00
  enso_oni 1980-01-31 00:00:00
  temperature_anomaly 1980-01-31 00:00:00
  precipitation_anomaly 1980-01-31 00:00:00
  drought_index 1980-01-31 00:00:00
  extreme_heat_events 1980-01-31 00:00:00
  disaster_event_count 1980-01-31 00:00:00
  disaster_storm_count 1980-01-31 00:00:00
  disaster_fire_count 1980-01-31 00:00:00
  disaster_flood_count 1980-01-31 00:00:00
Chosen panel start date: 1986-01-31 00:00:00
Saved /Users/sigmund/coding/github/EAS508-Statistics/data/raw/price_proxy_monthly.csv (1662, 7)
Saved /Users/sigmund/coding/github/EAS508-Statistics/data/raw/macro_monthly.csv (555, 6)
Saved /Users/sigmund/coding/github/EAS508-Statistics/data/raw/climate_disaster_monthly.csv (1662, 12)
Saved /Users/sigmund/coding/github/EAS508-Statistics/data/processed/monthly_panel.csv (1446, 22)


,date,price_proxy,commodity,price_source,price_source_detail,futures_price,futures_return,usd_index,usd_index_source,interest_rate,...,temperature_anomaly,precipitation_anomaly,drought_index,extreme_heat_events,enso_oni,disaster_event_count,disaster_storm_count,disaster_fire_count,disaster_flood_count,climate_location_code
72,1986-01-31,2.33,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.33,0.017316,64.9783,FRED_TWEXBMTH_then_TWEXBGSMTH,7.07,...,25.0,0.48,3.01,34.7,-0.49,0.0,0.0,0.0,0.0,261
73,1986-02-28,2.32,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.32,-0.004301,63.5747,FRED_TWEXBMTH_then_TWEXBGSMTH,7.06,...,23.2,1.69,3.21,31.4,-0.47,48.0,0.0,0.0,48.0,261
74,1986-03-31,2.29,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.29,-0.013015,62.9331,FRED_TWEXBMTH_then_TWEXBGSMTH,6.56,...,40.0,1.81,2.52,50.8,-0.31,21.0,0.0,0.0,21.0,261
75,1986-04-30,2.30,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.30,0.004357,62.7122,FRED_TWEXBMTH_then_TWEXBGSMTH,6.06,...,51.4,3.60,2.46,63.6,-0.20,1.0,0.0,0.0,0.0,261
76,1986-05-31,2.39,corn,USDA_NASS_PRICE_RECEIVED,CORN,2.39,0.038384,61.8118,FRED_TWEXBMTH_then_TWEXBGSMTH,6.15,...,60.7,4.06,2.33,71.9,-0.12,26.0,0.0,1.0,25.0,261
